In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter

In [ ]:
hollymac_data = pd.read_excel('HollyMcNamara_GoalsBySeason.xlsx')

In [ ]:
def extract_start_year(season_str):
    if isinstance(season_str, str) and '-' in season_str:
        return int(season_str.split('-')[0])
    return int(season_str)

In [ ]:
hollymac_data['club_debutseason'] = hollymac_data['season'].apply(extract_start_year)

In [ ]:
debut_season = hollymac_data['club_debutseason'].min()
hollymac_data['duration'] = hollymac_data['club_debutseason'] - debut_season

In [ ]:
hollymac_data['injured'] = hollymac_data['injured'].astype(bool)

In [ ]:
hollymac_data['event'] = 0

In [ ]:
covariates = ['matches', 'goals', 'injured']

In [ ]:
hollymac_data = hollymac_data.sort_values('club_debutseason')

In [ ]:
hollymac_data['cumulative_matches'] = hollymac_data['matches'].cumsum()
hollymac_data['cumulative_goals'] = hollymac_data['goals'].cumsum()
hollymac_data['cumulative_injuries'] = 3

In [ ]:
hollymac_data['event'] = 0

In [ ]:
survival_data = hollymac_data[['duration', 'event', 'cumulative_matches', 'cumulative_goals', 'cumulative_injuries']].dropna().tail(1)

In [ ]:
cph = CoxPHFitter()
cph.fit(survival_data, duration_col = 'duration', event_col = 'event')

In [ ]:
print("Cox Proportional Hazards Model Summary:")
cph.print_summary()

In [ ]:
median_survival = cph.median_survival_time
predicted_season = debut_season + median_survival - 1
print(f"\nPredicted season for Holly McNamara's first international goal (median): {round(predicted_season)}")

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(hollymac_data['duration'], event_observed = hollymac_data['event'], label = 'First international goal')
kmf.plot_survival_function()
plt.title("Survival curve showing time until Holly McNamara's first international goal")
plt.xlabel("Seasons since debut")
plt.ylabel("Survival probability (no goal scored yet)")
plt.show()